# Chapter 7 — Advanced Text Generation Techniques and Tools

**Book:** Hands-On Large Language Models  
**Chapter:** 7 — Going beyond prompt engineering with LangChain

---

## What You Will Learn

Chapter 6 showed what a single well-crafted prompt can do. This chapter introduces the machinery that lets you build **stateful, multi-step, tool-using** LLM applications:

- **LangChain chains**: compose prompts and models into reusable pipelines using the `|` operator
- **Multiple chains**: pipe the output of one chain as input to the next (title → character → story)
- **Memory**: give the model a persistent sense of conversation history across turns
  - `ConversationBufferMemory` — keeps the full history
  - `ConversationBufferWindowMemory` — keeps only the last k turns
  - `ConversationSummaryMemory` — compresses history into a rolling summary
- **Agents**: let the model decide which tool to call and when, using the ReAct reasoning pattern

---

## Parts Overview

| Part | Topic |
|------|-------|
| 1 | Loading the model via llama-cpp + LangChain |
| 2 | Single chains and prompt templates |
| 3 | Multiple chained LLM calls |
| 4 | Memory: Buffer, Window, Summary |
| 5 | ReAct agents with tools |

---

# Part 1 — Loading the Model

This chapter uses **llama-cpp-python** via LangChain's `LlamaCpp` wrapper. Unlike HuggingFace transformers, llama-cpp loads models in **GGUF** format (a quantisation-friendly binary format) and runs them using C++ kernels with optional GPU offloading via CUDA.

We download Phi-3-mini-4k-instruct in fp16 GGUF format from HuggingFace. The `-1` value for `n_gpu_layers` offloads all layers to GPU.

In [ ]:
# %%capture
# !pip install langchain>=0.1.17 langchain_community transformers>=4.40.1 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

In [ ]:
# Download the GGUF model file
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
from langchain import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

print("Model loaded")

In [ ]:
# Basic sanity check
print(llm.invoke("Hi! My name is Maarten. What is 1 + 1?"))

---

# Part 2 — Single Chains and Prompt Templates

## 2.1 PromptTemplate and the `|` Operator

A **PromptTemplate** is a reusable prompt with named variables (`{variable_name}`) that get filled in at runtime. Instead of hardcoding the full prompt string every time, you define a template once and call it with different inputs.

LangChain uses the **pipe operator** (`|`) to compose components into a chain. Writing `prompt | llm` creates a `RunnableSequence`: the output of `prompt.format(...)` (a string) becomes the input to `llm.invoke(...)`. This is LangChain's **LCEL** (LangChain Expression Language).

```
Input dict
    │
    ▼
PromptTemplate  ← fills {variable} placeholders
    │   (produces a formatted string)
    ▼
LlamaCpp LLM   ← generates text continuation
    │   (produces a string)
    ▼
Output string
```

In [ ]:
from langchain import PromptTemplate

# The Phi-3 chat template — {input_prompt} will be replaced at runtime
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

# Combine into a chain with the | operator
basic_chain = prompt | llm

print("Chain created:", type(basic_chain))

In [ ]:
# Invoke the chain with a dict of variables
result = basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
print(result)

### Theory Exercise — Manual Chain Trace

Before running a chain, you can inspect what the prompt template produces by calling `.format()` directly. This is useful for debugging — it shows you exactly what string gets sent to the model.

Below: format the template manually with your own input, then verify it matches the expected Phi-3 chat format.

In [ ]:
# Manually format the prompt template and print the result
# Use prompt.format(input_prompt="What is the capital of France?")
# YOUR CODE HERE
formatted_prompt = ""

print("Formatted prompt sent to model:")
print(repr(formatted_prompt))  # repr shows \n as actual newlines

---

# Part 3 — Multiple Chained LLM Calls

## 3.1 Building a Story Generation Pipeline

A **multi-chain pipeline** passes the output of one LLM call as a named variable into the next. Each step refines or extends the previous result.

We build a 3-step story generator:

```
summary (user input)
    │
    ▼
[title chain]       ← generates: title
    │   output_key = "title"
    ▼
[character chain]   ← receives: summary + title → generates: character
    │   output_key = "character"
    ▼
[story chain]       ← receives: summary + title + character → generates: story
```

Each `LLMChain` has an `output_key` that names its result. The pipe `|` operator forwards the full accumulated dict at each step, so later chains can access earlier outputs.

In [ ]:
from langchain import LLMChain

# Chain 1: Generate a story title from the summary
title_template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=title_template, input_variables=["summary"])
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# Test chain 1 in isolation
print(title_chain.invoke({"summary": "a girl that lost her mother"}))

In [ ]:
# Chain 2: Generate the main character using summary + title
character_template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=character_template, input_variables=["summary", "title"]
)
character_chain = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

print("Character chain created")

In [ ]:
# Chain 3: Generate the full story using summary + title + character
story_template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=story_template, input_variables=["summary", "title", "character"]
)
story_chain = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

print("Story chain created")

In [ ]:
# Combine all three chains with the pipe operator
# Each chain's output_key feeds into the next chain's input_variables
llm_chain = title_chain | character_chain | story_chain

# Run the full pipeline
result = llm_chain.invoke("a girl that lost her mother")

print("Title:", result.get("title", "").strip())
print()
print("Character:", result.get("character", "").strip())
print()
print("Story:", result.get("story", "").strip())

### Theory Exercise — Design a 4th Chain

Extend the pipeline by adding a 4th chain that takes the `story` output and generates a **one-sentence book blurb** — the kind you'd see on the back of a novel.

Requirements:
- The blurb must be exactly one sentence
- It must reference the main character's name (infer it from the `character` output)
- The `output_key` for this chain should be `"blurb"`

In [ ]:
# Chain 4: Generate a one-sentence blurb from the story
# YOUR CODE HERE
blurb_template = ""
blurb_prompt = None
blurb_chain = None

# full_pipeline = title_chain | character_chain | story_chain | blurb_chain
# result = full_pipeline.invoke("a boy who discovers he can talk to animals")
# print("Blurb:", result.get("blurb", "").strip())

---

# Part 4 — Memory

By default, each call to `llm.invoke()` is stateless — the model has no memory of previous turns. To build chatbots, assistants, or any multi-turn system, we need to maintain **conversation history** and inject it into each new prompt.

LangChain provides three memory strategies, each with a different trade-off between **token cost** and **information retention**:

| Memory Type | What It Stores | Token Cost | Information Loss |
|-------------|---------------|------------|------------------|
| `ConversationBufferMemory` | Full history verbatim | Grows with conversation | None |
| `ConversationBufferWindowMemory` | Last k turns only | Bounded | Loses older turns |
| `ConversationSummaryMemory` | Compressed summary | Sub-linear | Lossy compression |

The memory object injects conversation history into the prompt via a `{chat_history}` variable.

In [ ]:
# First: demonstrate the stateless problem
# Ask two separate questions — the second has no context from the first
result1 = basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
print("Turn 1:", result1)

result2 = basic_chain.invoke({"input_prompt": "What is my name?"})
print("Turn 2:", result2)

## 4.1 ConversationBufferMemory

`ConversationBufferMemory` appends every user message and assistant response to a running transcript. Before each new generation, the full transcript is injected into the `{chat_history}` slot of the prompt.

The prompt template must include `{chat_history}` as a variable for this to work.

In [ ]:
# Updated prompt template with {chat_history} slot
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [ ]:
from langchain.memory import ConversationBufferMemory

# Create memory instance — memory_key must match the template variable
memory = ConversationBufferMemory(memory_key="chat_history")

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory,
)

print("Chain with buffer memory created")

In [ ]:
# Turn 1: introduce name
result = llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
print("Turn 1:", result["text"])

In [ ]:
# Turn 2: does the model remember the name?
result = llm_chain.invoke({"input_prompt": "What is my name?"})
print("Turn 2:", result["text"])

### Theory Exercise — Inspect Memory Contents

After two turns, inspect what the memory object actually stores. Call `memory.load_memory_variables({})` and print the result. Then explain:

1. What format is the history stored in?
2. How many tokens do you estimate this adds to every subsequent prompt?
3. If you ran 100 turns, what problem would `ConversationBufferMemory` have?

In [ ]:
# Inspect what the memory object holds after 2 turns
# YOUR CODE HERE
memory_contents = None  # replace with memory.load_memory_variables({})
print(memory_contents)

## 4.2 ConversationBufferWindowMemory

`ConversationBufferWindowMemory` keeps only the **last k conversation turns**. Older turns are dropped. This bounds the token cost at the expense of losing early context.

Setting `k=2` means the model sees at most 2 prior exchanges (2 user messages + 2 assistant responses) regardless of how long the conversation has been.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# k=2 means retain only the last 2 conversation pairs
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory,
)

print("Window memory (k=2) chain created")

In [ ]:
# Turn 1: introduce name and age
result = llm_chain.invoke({"input_prompt": "Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
print("Turn 1:", result["text"])

# Turn 2: a filler question that pushes Turn 1 toward the window edge
result = llm_chain.invoke({"input_prompt": "What is 3 + 3?"})
print("Turn 2:", result["text"])

In [ ]:
# Turn 3: ask for name — is it still in the k=2 window?
result = llm_chain.invoke({"input_prompt": "What is my name?"})
print("Turn 3 (name query):", result["text"])

In [ ]:
# Turn 4: ask for age — was it evicted from the window?
result = llm_chain.invoke({"input_prompt": "What is my age?"})
print("Turn 4 (age query):", result["text"])

### Theory Exercise — Window Size Experiment

Run the same conversation again but with `k=1`. Predict which facts the model will and won't remember before running the cells.

Then check: does the model remember the name after Turn 3 with `k=1`? Why or why not?

In [ ]:
# Experiment: set k=1 and repeat the same 4-turn conversation
# YOUR CODE HERE
memory_k1 = None  # ConversationBufferWindowMemory(k=1, memory_key="chat_history")
llm_chain_k1 = None  # LLMChain(prompt=prompt, llm=llm, memory=memory_k1)

## 4.3 ConversationSummaryMemory

`ConversationSummaryMemory` uses the LLM itself to **compress the conversation into a running summary**. After each turn, the model is prompted to update the summary with the latest exchange.

This approach:
- Keeps token count roughly constant regardless of conversation length
- Preserves high-level facts (names, topics, decisions) in compressed form
- Loses verbatim details and exact phrasing

The summary prompt is also a template — it takes the current summary and the new turn as inputs and outputs an updated summary.

In [ ]:
# Summary prompt: given current summary + new conversation lines, produce updated summary
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

print("Summary prompt template defined")

In [ ]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory,
)

print("Summary memory chain created")

In [ ]:
# Turn 1 + 2
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

In [ ]:
# Ask what the first question was — tests whether summary preserves it
result = llm_chain.invoke({"input_prompt": "What was the first question I asked?"})
print(result["text"])

In [ ]:
# Inspect the current summary stored in memory
print(memory.load_memory_variables({}))

### Theory Exercise — Memory Comparison

Complete the comparison table based on what you observed. Then answer: for which use case would each memory type be most appropriate?

| Memory Type | Remembers name after 4 turns? | Remembers age after window eviction? | Token growth | Extra LLM calls per turn |
|-------------|------------------------------|-------------------------------------|-------------|-------------------------|
| Buffer | | | Linear | 0 |
| Window (k=2) | | | Constant | 0 |
| Summary | | | Sub-linear | 1 |

**Best use cases:**
- Buffer: ?
- Window: ?
- Summary: ?

---

# Part 5 — Agents and the ReAct Pattern

## 5.1 What Is an Agent?

A **chain** has a fixed sequence of steps decided at design time. An **agent** lets the model decide at runtime which action to take next. This enables LLMs to use external tools — web search, calculators, databases, APIs — when the answer cannot be derived from the model's weights alone.

The **ReAct** (Reasoning + Acting) pattern structures this decision loop as alternating **Thought** and **Action** steps:

```
Question: What is the price of a MacBook Pro in EUR?

Thought: I need to find the current USD price first.
Action: duckduck["MacBook Pro price USD 2024"]
Observation: MacBook Pro starts at $1,599

Thought: Now I need to convert USD to EUR at 0.85 rate.
Action: Calculator[1599 * 0.85]
Observation: 1359.15

Thought: I now have both values.
Final Answer: A MacBook Pro costs $1,599 USD or approximately €1,359.15 EUR.
```

The model generates the Thought and Action, the framework executes the tool and injects the Observation, and the loop repeats until the model outputs `Final Answer:`.

**Note:** The Agents section uses OpenAI's GPT-3.5-turbo because local models may struggle to reliably follow the strict ReAct output format.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

print("OpenAI LLM ready")

In [ ]:
from langchain import PromptTemplate

# The ReAct prompt template — note the {tools}, {tool_names}, {input}, {agent_scratchpad} variables
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

print("ReAct prompt template defined")

In [ ]:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

# Tool 1: DuckDuckGo web search
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# Tool 2: LLM-based math calculator
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

print(f"Tools loaded: {[t.name for t in tools]}")

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent

# Build the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

print("Agent executor ready")

In [ ]:
# Run the agent on a question requiring both search and math
result = agent_executor.invoke(
    {
        "input": (
            "What is the current price of a MacBook Pro in USD? "
            "How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
        )
    }
)
print("Final Answer:", result["output"])

### Theory Exercise — Manual ReAct Trace

Before running the agent, write out the expected Thought/Action/Observation loop by hand for this question:

> "Who wrote the book 'Hands-On Large Language Models' and what year was it published? How many years ago was that?"

Fill in the template below with what you predict the agent will do. Then run it and compare.

In [ ]:
# Write your predicted ReAct trace as a multi-line string
predicted_trace = """
Question: Who wrote 'Hands-On Large Language Models' and what year was it published? How many years ago was that?

Thought: 
Action: 
Action Input: 
Observation: 

Thought: 
Action: 
Action Input: 
Observation: 

Thought: I now know the final answer.
Final Answer: 
"""
print("Predicted trace:")
print(predicted_trace)

In [ ]:
# Now run the agent on the same question and compare to your prediction
# YOUR CODE HERE
# result = agent_executor.invoke({"input": "Who wrote the book 'Hands-On Large Language Models' and what year was it published? How many years ago was that?"})
# print("Final Answer:", result["output"])

---

# Chapter Summary

## Key Concepts

**LangChain chains** compose prompts and models into reusable pipelines using the `|` operator (LCEL). Each component receives a dict and returns a dict, making it easy to pass named outputs from one step to the next.

**Multiple chains** enable complex pipelines where each LLM call refines or extends the previous result. The `output_key` of each `LLMChain` determines how its result is named in the accumulated dict passed downstream.

**Memory** solves the statelessness problem. The right memory type depends on the trade-off between token cost and information retention: Buffer is accurate but expensive; Window is bounded but forgets; Summary is compressed but lossy.

**ReAct agents** give the model agency over which tools to call and in what order. The Thought/Action/Observation loop is an explicit reasoning trace that makes the model's decision process inspectable. Local models struggle with this pattern because they must reliably parse and generate a strict format.

## When to Use Each Technique

```
Is your task a fixed, known sequence of steps?
├── YES → Use chains (prompt | llm or LLMChain)
│         Need to pass context between steps? → Multiple chained LLMChains
└── NO → Need to call external tools dynamically?
         ├── YES → Use ReAct agent (requires strong model — GPT-4 or similar)
         └── NO → Rethink — can you decompose into fixed steps?

Does your application need multi-turn conversation?
├── Short conversations, need full accuracy → ConversationBufferMemory
├── Long conversations, bounded token budget → ConversationBufferWindowMemory (k=5-10)
└── Very long conversations, key facts only → ConversationSummaryMemory
```